What happens when particles move stochastically? Every diffusion model (DDPM, DDIM, score SDEs) has Brownian motion at its core. The discrete forward chain you saw in the [DDPM tutorial](/aiml-common/lectures/diffusion/ddpm/ddpm_mog_tutorial), clusters dissolving into Gaussian noise, is a discrete variance-preserving process: each step shrinks the signal and adds fresh noise. Its continuous-time form is the Ornstein-Uhlenbeck (variance-preserving) SDE of the drift section below, driven by a Wiener process. Building intuition for the Wiener process makes the rest of the diffusion-model machinery click.

In the [probability transport chapter](/aiml-common/lectures/optimal-transport/dynamic-transport/index) each particle followed a velocity field, and its path was smooth. A Brownian path has no velocity: it is continuous but nowhere differentiable. Its density still changes smoothly in time. The continuity equation of the transport chapter describes how a velocity field moves a density; Brownian motion adds a spreading term to that equation, and the result is the Fokker-Planck equation.

## Math recap: the Wiener process

A standard Wiener process $W_t$ on $[0, T]$ is a continuous-time stochastic process satisfying:

* $W_0 = 0$
* Independent increments: for $0 \le s < t$, $W_t - W_s \sim \mathcal{N}(0, t - s)$, independent of the path up to time $s$
* Continuous paths

A discrete-time approximation with step $\Delta t$ is the **random walk**

$$ W_{t + \Delta t} = W_t + \sqrt{\Delta t}\; \xi, \quad \xi \sim \mathcal{N}(0, 1) $$

Multiplied by a noise scale $\sigma$ this gives Brownian motion with variance $\sigma^2 t$. Later sections write the noise scale as $g$, the same quantity as $\sigma$ here, and use the diffusion coefficient $D = g^2/2$ in the next two sections.

The DDPM forward chain $q(x_t \mid x_{t-1}) = \mathcal{N}(x_t; \sqrt{1-\beta_t}\, x_{t-1}, \beta_t I)$ is a *variance-preserving* discrete-time analogue: it adds Brownian-style noise *and* shrinks the signal each step so the marginal stays bounded.

## Brownian motion in motion

Eight particles start at the origin and trace independent random walks. The trail behind each particle is its history: short trajectories stay near the origin on either side, and the cloud widens the longer the walk runs.

<video
  src="https://artifacts.aegeanai.com/manim/brownian-motion/BrownianMotion.mp4"
  controls
  loop
  muted
  playsInline
  width="100%"
  style={{ borderRadius: 8 }}
/>

## Eight Brownian trajectories

Eight particles released from the origin, each undergoing an independent random walk over 500 steps with $\sigma = 0.7$ and $\Delta t = 1/60$. The colored end-dots mark each trajectory's final position; the black dot at the origin marks the common start.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
n_particles = 8
n_steps = 500
dt = 1.0 / 60
sigma = 0.7

increments = sigma * np.sqrt(dt) * rng.standard_normal((n_steps + 1, n_particles, 2))
increments[0] = 0.0
paths = np.cumsum(increments, axis=0)

In [ ]:
colors = plt.cm.tab10(np.arange(n_particles))
fig, ax = plt.subplots(figsize=(8, 8))
for i in range(n_particles):
    ax.plot(paths[:, i, 0], paths[:, i, 1], color=colors[i], alpha=0.7, lw=1.2)
    ax.scatter(paths[-1, i, 0], paths[-1, i, 1], color=colors[i], s=60, zorder=3,
               edgecolor="white", linewidth=1.0)
ax.scatter([0], [0], color="black", s=80, zorder=3, label="start (t=0)")
ax.set_aspect("equal")
ax.set_title(f"{n_particles} Brownian trajectories, $\\sigma=${sigma}, {n_steps} steps")
ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## Snapshot grid of the Wiener process

200 fresh particles released from the origin, snapshotted at $t = 0, 10, 50, 100, 200, 500$, the step indices the [DDPM tutorial](/aiml-common/lectures/diffusion/ddpm/ddpm_mog_tutorial) uses for its forward-noising visualization. These are Wiener snapshots at those indices, not DDPM marginals. The cloud spreads with variance proportional to $t$, whereas the DDPM marginal variance saturates, as the drift section below shows.

In [ ]:
rng = np.random.default_rng(0)
n = 200
T = 500
snapshots = [0, 10, 50, 100, 200, 500]
sigma_grid = 0.5

increments = sigma_grid * np.sqrt(dt) * rng.standard_normal((T + 1, n, 2))
increments[0] = 0.0
grid_paths = np.cumsum(increments, axis=0)

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(24, 4))
for ax, t in zip(axes, snapshots):
    pts = grid_paths[min(t, T)]
    ax.scatter(pts[:, 0], pts[:, 1], s=12, alpha=0.55, color="C0")
    ax.set_title(f"t = {t}")
    ax.set_xlim(-8, 8); ax.set_ylim(-8, 8); ax.set_aspect("equal")
    ax.grid(alpha=0.3)
fig.suptitle("Brownian-motion snapshot grid - variance grows linearly with t")
plt.show()

## Diffusion flux from unbiased steps

Put particles on a line with spacing $\Delta x$. In each time step $\Delta t$, every particle moves one site to the left or one site to the right, with probability one half each. The mean step is zero, so no particle prefers a direction.

Now count the particles that cross the bond between site $j$ and site $j+1$ in one step. Of the $n_j$ particles at site $j$, about $n_j/2$ step right. Of the $n_{j+1}$ particles at site $j+1$, about $n_{j+1}/2$ step left. On average, the net number crossing from left to right is

$$
\tfrac{1}{2}\,(n_j - n_{j+1}).
$$

Where the density falls from left to right, $n_j > n_{j+1}$ and the net crossing is positive. Mass moves down the gradient even though every step is unbiased.

To take the continuum limit, write $n_j = N\,p(x_j)\,\Delta x$ for $N$ particles with density $p$. For small $\Delta x$ the net crossing per step becomes $-\tfrac{1}{2}\,N\,\Delta x^2\,\partial_x p$. Each step has variance $\Delta x^2$. Brownian motion with noise scale $g$ (the $\sigma$ above) has $\Delta x^2 = g^2 \Delta t$, so $\Delta x^2 / (2\Delta t) = g^2/2 = D$, and the net crossing per step is $N\,(-D\,\partial_x p)\,\Delta t$. The diffusion flux is therefore Fick's law,

$$
J_{\text{diff}} = -D\,\partial_x p, \qquad D = \frac{g^2}{2}.
$$

Take $N = 10^9$ particles started from a smooth, decreasing profile (a logistic step on $[-8, 8]$) with $D = 1$ and $\Delta x = 0.1$, so that $\Delta t = \Delta x^2 / (2D) = 0.005$. Draw the random steps for every bond and compare the net crossing count with $-D\,\partial_x p\,\Delta t\,N$.

In [ ]:
D_coef = 1.0                      # diffusion coefficient, D = g^2 / 2
lat_dx = 0.1                      # lattice spacing
lat_dt = lat_dx**2 / (2 * D_coef) # time step that matches D
n_total = 10**9                   # number of particles
profile_width = 1.0
half_len = 8.0

sites = np.arange(-half_len, half_len + lat_dx / 2, lat_dx)

def profile_unnormalized(x):
    return 1.0 / (1.0 + np.exp(x / profile_width))

def profile_unnormalized_dx(x):
    e = np.exp(x / profile_width)
    return -(e / (1.0 + e) ** 2) / profile_width

# Normalize on the interval [-half_len, half_len]
fine = np.linspace(-half_len, half_len, 200001)
Z = np.trapezoid(profile_unnormalized(fine), fine)

def profile(x):
    return profile_unnormalized(x) / Z

def profile_dx(x):
    return profile_unnormalized_dx(x) / Z

# Particle counts per site and one step of unbiased random walk
counts = np.rint(n_total * profile(sites) * lat_dx).astype(np.int64)
rng_flux = np.random.default_rng(0)
go_right = rng_flux.binomial(counts, 0.5)         # steps right from each site
go_left = counts - go_right                       # steps left from each site

# Net crossings of the bond between site j and site j+1, left to right
bond_x = 0.5 * (sites[:-1] + sites[1:])
net_all = go_right[:-1] - go_left[1:]
fick_all = -D_coef * profile_dx(bond_x) * lat_dt * n_total
se_all = np.sqrt((counts[:-1] + counts[1:]) / 4.0)

# Three interior probes. Each probe is a bond between two lattice sites,
# so it sits half a spacing from the round value in its label.
probe_x = np.array([-2.0, 0.0, 2.0])
probe_idx = np.array([np.argmin(np.abs(bond_x - x)) for x in probe_x])
net_crossings = net_all[probe_idx]
fick_prediction = fick_all[probe_idx]
crossing_se = se_all[probe_idx]

for x, a, b, s in zip(bond_x[probe_idx], net_crossings, fick_prediction, crossing_se):
    print(f"x = {x:+.2f}   net crossings = {a:>10d}   Fick = {b:>12.1f}   se = {s:6.1f}")

In [ ]:
# Fick's law: net crossings per step against -D dp/dx dt, at three probe points
assert np.all(np.abs(net_crossings - fick_prediction) < 0.05 * np.abs(fick_prediction) + 3 * crossing_se)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
mask = np.abs(bond_x) <= 6
ax.plot(bond_x[mask], net_all[mask] / n_total, lw=0.8, color="C0", label="net crossings per particle")
ax.plot(bond_x[mask], fick_all[mask] / n_total, lw=2.0, ls="--", color="C3", label=r"$-D\,\partial_x p\,\Delta t$")
ax.scatter(bond_x[probe_idx], net_crossings / n_total, color="k", zorder=3, label="probe points")
ax.set_xlabel("x")
ax.set_ylabel("net left-to-right crossings per particle")
ax.set_title("Unbiased steps produce a net flux down the density gradient")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

At all three probe points the net crossing count matches Fick's law to within a few percent. Along the whole profile in the figure, the net count follows $-D\,\partial_x p\,\Delta t$ and goes to zero where the density is flat.

## Adding a drift

A drift adds a second way to move mass. The stochastic differential equation

$$
dX_t = f(X_t)\,dt + g\,dW_t
$$

moves each particle by $f\,dt$ on average, on top of the random part. The drift carries density with the local velocity $f$, so it adds the flux $J_{\text{drift}} = f\,p$. The total flux is the sum of the two terms,

$$
J = f\,p - D\,\partial_x p.
$$

The next section turns this flux into an equation for $p_t$. First, the drift acting on paths.

The running example uses $f(x) = -x$ and $g = \sqrt{2}$, so $D = 1$. This is the Ornstein-Uhlenbeck process. Started from a point $x_0$, its position at time $t$ is Gaussian with mean $x_0\,e^{-t}$ and variance $1 - e^{-2t}$. The mean decays to zero and the variance rises to 1, which is the variance of $\mathcal{N}(0, 1)$. The position $X_t$ has the same distribution as $e^{-t} x_0 + \sqrt{1 - e^{-2t}}\,Z$ with $Z \sim \mathcal{N}(0, 1)$ at each fixed $t$. This holds at each fixed $t$ separately: reusing a single $Z$ for every $t$ would not give Ornstein-Uhlenbeck paths. Paths are built step by step from the exact transition

$$
X_{t+h} = e^{-h}\,X_t + \sqrt{1 - e^{-2h}}\,Z.
$$

Simulate 20000 paths from $x_0 = 2$ and compare the sample mean and variance at four times with the exact values.

In [ ]:
x_start = 2.0
ou_h = 0.02
ou_horizon = 4.0
ou_paths = 20000
ou_steps = int(round(ou_horizon / ou_h))
t_snap = np.array([0.5, 1.0, 2.0, 4.0])

rng_ou = np.random.default_rng(0)
ou_x = np.empty((ou_steps + 1, ou_paths))
ou_x[0] = x_start
decay = np.exp(-ou_h)
noise_sd = np.sqrt(1.0 - np.exp(-2.0 * ou_h))
for k in range(ou_steps):
    ou_x[k + 1] = decay * ou_x[k] + noise_sd * rng_ou.standard_normal(ou_paths)

snap_idx = np.rint(t_snap / ou_h).astype(int)
sample_mean = ou_x[snap_idx].mean(axis=1)
sample_var = ou_x[snap_idx].var(axis=1, ddof=1)
exact_var = 1.0 - np.exp(-2.0 * t_snap)
mean_se = np.sqrt(exact_var / ou_paths)
var_se = exact_var * np.sqrt(2.0 / (ou_paths - 1))

for t, m, v, em, ev in zip(t_snap, sample_mean, sample_var, x_start * np.exp(-t_snap), exact_var):
    print(f"t = {t:.1f}   mean {m:+.4f} (exact {em:+.4f})   variance {v:.4f} (exact {ev:.4f})")

In [ ]:
# Ornstein-Uhlenbeck moments at the snapshot times
assert np.all(np.abs(sample_mean - x_start * np.exp(-t_snap)) < 4 * mean_se)
assert np.all(np.abs(sample_var - (1 - np.exp(-2 * t_snap))) < 4 * var_se)

In [ ]:
t_axis = np.arange(ou_steps + 1) * ou_h
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(t_axis, ou_x[:, :30], color="C0", alpha=0.35, lw=0.8)
mean_curve = x_start * np.exp(-t_axis)
sd_curve = np.sqrt(1.0 - np.exp(-2.0 * t_axis))
ax.plot(t_axis, mean_curve, color="k", lw=2.0, label=r"mean $x_0 e^{-t}$")
ax.fill_between(t_axis, mean_curve - sd_curve, mean_curve + sd_curve, color="C3", alpha=0.15,
                label=r"mean $\pm$ one standard deviation")
ax.set_xlabel("t")
ax.set_ylabel("X(t)")
ax.set_title("Ornstein-Uhlenbeck paths from x = 2")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

The paths start at $x_0 = 2$ and scatter around a mean that decays like $e^{-t}$. The spread widens at first, like Brownian motion, and then stops growing because the drift pulls paths back toward zero. By $t = 4$ the sample variance is close to 1 and the mean is close to 0.

## Connection back to diffusion

Continuous-time diffusion frames the forward process as the SDE

$$ dx_t = f(x_t, t)\, dt + g(t)\, dW_t $$

where $W_t$ is exactly the Brownian motion above. The variance-preserving SDE used by DDPM picks $f(x_t, t) = -\tfrac{1}{2}\beta(t)\, x_t$ and $g(t) = \sqrt{\beta(t)}$, so the variance stays bounded. Only the transition $q(x_t \mid x_0)$ is Gaussian. The marginal $p_t$ remains a mixture at every finite $t$ and approaches a Gaussian only as $t$ grows.

The discrete chain of the DDPM tutorial is the exact Ornstein-Uhlenbeck transition of the drift section, not an Euler-Maruyama approximation of it. A DDPM step $\sqrt{1-\beta_k}\, X_{k-1} + \sqrt{\beta_k}\, \epsilon_k$ equals $e^{-h_k} X_{k-1} + \sqrt{1 - e^{-2h_k}}\, \epsilon_k$ with $h_k = -\tfrac{1}{2}\log(1 - \beta_k)$. Step $k$ therefore sits at Ornstein-Uhlenbeck time $t_k = -\tfrac{1}{2}\log \bar\alpha_k$, where $\bar\alpha_k = \prod_{j \le k} (1 - \beta_j)$.

The forward process is Brownian motion plus a drift. The reverse direction uses the score. The network $\epsilon_\theta$ estimates $\mathbb{E}[\epsilon \mid x_k]$, and the score at step $k$ is $\partial_x \log p_{t_k}(x_k) = -\mathbb{E}[\epsilon \mid x_k] / b_k$ with $b_k = \sqrt{1 - \bar\alpha_k}$. This $b_k$ equals the Ornstein-Uhlenbeck $b_t = \sqrt{1 - e^{-2t}}$ at $t = t_k$. The reverse-time drift follows from this score, and deterministic DDIM is the probability flow ODE limit.

The two sections above give the total flux $J = f\,p - D\,\partial_x p$ with $D = g^2/2$: a drift term that carries density and a diffusion term that spreads it. The next section, [Fokker-Planck and the probability flow ODE](/aiml-common/lectures/diffusion/fokker-planck/fokker_planck), uses conservation of mass to turn $J$ into the Fokker-Planck equation.

## References

1. Anderson. *Reverse-time diffusion equation models.* Stochastic Processes Appl. 1982.
2. Song et al. *Score-Based Generative Modeling through Stochastic Differential Equations.* ICLR 2021. [arxiv.org/abs/2011.13456](https://arxiv.org/abs/2011.13456)
3. Karatzas & Shreve. *Brownian Motion and Stochastic Calculus.* Springer 1991. (textbook)